In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

## 1.基础的版本

In [2]:
class BasicExpert(nn.Module):
    def __init__(self,feature_in,feature_out):
        super().__init__()
        self.fc = nn.Linear(feature_in,feature_out)
    
    def forward(self,x):
        return self.fc(x)
    

In [ ]:
class BasicMoe(nn.Module):
    def __init__(self, feature_in,feature_out,num_experts):
        super().__init__()
        self.gate = nn.Linear(feature_in,num_experts)
        # output shape [batch_size,num_experts]
        self.experts = nn.ModuleList(
            [BasicExpert(feature_in,feature_out) for _ in range(num_experts)]
        )

    def forward(self,x):
        # x.shape [batch_size,feature_in]
        # feature_in 也叫做 hidden_size,hidden_dim
        expert_weights = self.gate(x) #[batch_size,num_experts]
        expert_out_list = [
            expert(x) for expert in self.experts
        ] #每一个expert 输出都是[batch,feature_out]

        expert_output = torch.concat(
            [expert_out.unsqueeze(1) for expert_out in expert_out_list],
            # expert_out shape [batch_size,1,feature_out]
            dim = 1
        ) # expert_output shape [batch_size,num_experts,feature_out]

        expert_weights = F.softmax(expert_weights,dim=-1) 
        #[batch_size,num_experts]

        # 希望的输出是 [batch_size,feature_out]
        expert_weights = expert_weights.unsqueeze(1) #shape [batch_size,1,num_experts]

        output = expert_weights@expert_output #[batch_size,1,feature_out]

        return output.squeeze(1)

## 验证
x = torch.rand(2,10)
moe = BasicMoe(10,5,5)
y = moe(x)
print(y.shape)


torch.Size([2, 5])


## 2. SparseMoE,MOE LLM 


In [ ]:
from dataclasses import dataclass
class MOEConfig:
    def __init__(self,
                 hidden_dim,
                 expert_number,
                 top_k,
                 shared_experts_number=2):
        self.hidden_dim = hidden_dim
        self.expert_number = expert_number
        self.top_k = top_k
        self.shared_experts_number = shared_experts_number

class MOERounter(nn.Module):
    def __init__(self,config):
       self.gate = nn.Linear(config.hidden_dim,config.expert_number)
       self.expert_number = config.expert_number
       self.top_k = config.top_k

    def forward(self,x):
        # x.shape [batch_size*seq_len,hidden_dim]
        router_logits = self.gate(x) #[batch_size*seq_len,expert_number]

        # 计算每一个专家的概率
        router_porbs = F.softmax(router_logits,dim=-1,dtype=torch.float)

        
        router_weights,selected_experts_indices = torch.topk(
            router_porbs,
            self.top_k,
            dim=-1
        ) # router_weights.shape [batch_size*seq_len,top_k]

        # 
        router_weights = router_weights/router_weights.sum(dim=-1,keepdim=True)
        router_weights = router_weights.to(x.dtype)

        expert_mask = F.one_hot(
            selected_experts_indices,
            num_classes=self.expert_number,
        )# [batch*seq_len,topk,expert_number]


class SparseMOE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.top_k = config.top_k
        self.hidden_dim = config.hidden_dim
        self.expert_number = config.expert_number

        # 初始化专家
        self.experts = nn.ModuleList(
            BasicExpert(config.hidden_dim,config.hidden_dim)
            for _ in range(config.expert_number)
        )

        self.router = None #TODO
    
    def forward(self,x): # x.shape [batch,seq_len,hidden_dim]


    

In [8]:
import torch
a = torch.rand(1,5,size=(2,3,4))
a

TypeError: rand() received an invalid combination of arguments - got (int, int, size=tuple), but expected one of:
 * (tuple of ints size, *, torch.Generator generator, tuple of names names, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
 * (tuple of ints size, *, torch.Generator generator, Tensor out, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
 * (tuple of ints size, *, Tensor out, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
 * (tuple of ints size, *, tuple of names names, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)


In [9]:
a.pow(2).mean(dim=-1,keepdim=True)

RuntimeError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Long